# 24. Normalization Test (Pipeline 05)

- Goal: check body-relative normalization and confirm only `norm` coordinates are produced here.
- Docs: `docs_eng/pipeline/05_normalization.md` / `docs/pipeline/05_normalization.md`
- Inputs: Preprocessed pose dataframe from prior-stage setup.
- Outputs: In-memory normalized dataframe/report unless a cell explicitly saves under `data/processed/`.
- Checks: direct normalization output, provenance preservation, visual comparison, and pipeline integration.


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import json
import warnings

import pandas as pd

from movement.config import CONNECTIONS, LANDMARKS
from movement.normalization import (
    check_normalization_result,
    normalize_pose_by_hip_torso,
)
from movement.pipeline import (
    NormalizationConfig,
    PreprocessingConfig,
    run_pipeline,
)
from movement.stage_context import (
    build_stage_check_pipeline_config,
    prepare_previous_stage_inputs,
)
from movement.visualization import create_pose_comparison_animation

print('imports OK')


## Data Setup

Runs the prior context in pipeline order before normalization:

```text
Pose CSV -> ① Validation -> ② Annotation -> ③ Exercise Definition -> ④ Preprocessing -> ⑤ Normalization
```


In [ ]:
pose_csv = 'data/pose/mediapipe/no_consent/20260517/p01_squat_set1_output_pose.csv'
annotation_csv = 'data/pose/mediapipe/no_consent/20260517/p01_squat_set1_annotation.csv'
TARGET_EXERCISE_ID = 'squat'


def estimate_frame_duration_ms(dataframe, default_ms=33):
    if 'timestamp' not in dataframe.columns:
        return default_ms
    dt = dataframe['timestamp'].astype(float).diff().dropna()
    if dt.empty:
        return default_ms
    median_dt = float(dt.median())
    if median_dt <= 0:
        return default_ms
    return max(1, int(round(median_dt * 1000)))


pre_config = PreprocessingConfig(enabled=True)
stage_inputs = prepare_previous_stage_inputs(
    prepare_until='preprocessing',
    pose_csv=pose_csv,
    annotation_csv=annotation_csv,
    exercise_id=TARGET_EXERCISE_ID,
    landmarks=LANDMARKS,
    preprocessing_config=pre_config,
)

df_raw = stage_inputs.raw_df
df_annotated = stage_inputs.annotated_df
val_report = stage_inputs.validation_report
ann_report = stage_inputs.annotation_report
exercise_def = stage_inputs.exercise_definition
pre_df = stage_inputs.preprocessed_df
pre_report = stage_inputs.preprocessing_report
TARGET_DEFINITIONS_DIR = stage_inputs.definitions_dir
frame_duration_ms = estimate_frame_duration_ms(pre_df)

setup_summary = pd.DataFrame([
    {'item': 'frames_loaded', 'value': len(df_raw)},
    {'item': 'validation_passed', 'value': val_report['passed']},
    {'item': 'structural_validation_passed', 'value': val_report.get('structural_passed')},
    {'item': 'analysis_frames', 'value': f"{ann_report['num_analysis_frames']} / {ann_report['num_total_frames']}"},
    {'item': 'exercise_id', 'value': exercise_def.exercise_id},
    {'item': 'movement_template_id', 'value': exercise_def.classification['movement_template_id']},
    {'item': 'preprocessing_invalid_frames', 'value': pre_report['num_invalid_frames']},
    {'item': 'preprocessed_shape', 'value': pre_df.shape},
    {'item': 'definitions_dir', 'value': str(TARGET_DEFINITIONS_DIR)},
])
display(setup_summary)

if val_report.get('warnings'):
    display(pd.DataFrame(val_report['warnings']))
if not val_report['passed']:
    print('NOTE: structural validation failed; inspect before normalization.')


## Direct Normalization Test


In [ ]:
norm_config = NormalizationConfig(
    enabled=True,
    keep_reference_columns=True,
    model_depth_scale=1.0,
)

norm_df, norm_report = normalize_pose_by_hip_torso(
    df=pre_df,
    landmarks=LANDMARKS,
    keep_reference_columns=norm_config.keep_reference_columns,
    model_depth_scale=norm_config.model_depth_scale,
)

print(f'normalized shape: {norm_df.shape}')
print(json.dumps(norm_report, indent=2, ensure_ascii=False))


## Check 1: Normalized Output Columns and Provenance


In [ ]:
required_preprocessing_columns = ['preprocessing_valid']
optional_preprocessing_columns = ['preprocessing_confidence']
emitted_optional_preprocessing_columns = [
    col for col in optional_preprocessing_columns if col in pre_df.columns
]
usable_columns = [col for col in pre_df.columns if col.endswith('_usable')]
source_columns = [col for col in pre_df.columns if col.endswith('_preprocessing_source')]

expected_norm_columns = [
    f'{lm}_{axis}'
    for lm in LANDMARKS
    for axis in ('norm_x', 'norm_y', 'norm_z')
]
missing_norm_columns = [col for col in expected_norm_columns if col not in norm_df.columns]
assert not missing_norm_columns, f'missing norm columns: {missing_norm_columns[:10]}'

preserved_preprocessing_columns = (
    required_preprocessing_columns
    + emitted_optional_preprocessing_columns
    + usable_columns[:3]
    + source_columns[:3]
)
missing_preserved_columns = [
    col for col in preserved_preprocessing_columns if col not in norm_df.columns
]
assert not missing_preserved_columns, f'normalization dropped preprocessing provenance: {missing_preserved_columns}'

reference_columns = [
    'hip_center_x', 'hip_center_y', 'hip_center_z',
    'shoulder_center_x', 'shoulder_center_y', 'shoulder_center_z',
    'torso_length',
]
missing_reference_columns = [col for col in reference_columns if col not in norm_df.columns]
assert not missing_reference_columns, f'missing reference columns: {missing_reference_columns}'

unexpected_coordinate_families = [
    col for col in norm_df.columns if '_canon_' in col or '_candidate_' in col
]
assert not unexpected_coordinate_families, f'unexpected downstream coordinate columns: {unexpected_coordinate_families[:10]}'

contract_summary = pd.DataFrame([
    {'item': 'norm_coordinate_columns_present', 'value': len(expected_norm_columns)},
    {'item': 'preprocessing_valid_preserved', 'value': 'preprocessing_valid' in norm_df.columns},
    {'item': 'landmark_usable_columns_preserved', 'value': len(usable_columns)},
    {'item': 'landmark_source_columns_preserved', 'value': len(source_columns)},
    {'item': 'reference_columns_preserved', 'value': True},
    {'item': 'unexpected_downstream_coordinate_columns', 'value': len(unexpected_coordinate_families)},
])
display(contract_summary)
print('PASS: normalization output columns and preprocessing provenance are valid')


## Check 2: Normalization Summary and Configuration


In [ ]:
check_report = check_normalization_result(norm_df)
assert check_report['passed'] is True

norm_columns = [col for col in norm_df.columns if col.endswith('_norm_x')]
normalization_stage_summary = pd.DataFrame([
    {'item': 'exercise_id', 'value': exercise_def.exercise_id},
    {'item': 'input_dataframe', 'value': 'pre_df from preprocessing'},
    {'item': 'input_frames', 'value': len(pre_df)},
    {'item': 'normalization_method', 'value': norm_report['method']},
    {'item': 'normalization_scale_value', 'value': round(float(norm_report['scale_value']), 6)},
    {'item': 'invalid_torso_frames', 'value': norm_report['num_invalid_torso_frames']},
    {'item': 'model_depth_scale', 'value': norm_report['model_depth_scale']},
    {'item': 'max_abs_hip_center', 'value': check_report['max_abs_hip_center']},
    {'item': 'median_normalized_torso_length', 'value': round(float(check_report['median_normalized_torso_length']), 6)},
    {'item': 'norm_x_column_count', 'value': len(norm_columns)},
])
display(normalization_stage_summary)

active_normalization_config = pd.DataFrame([
    {'area': 'normalization', 'setting': 'enabled', 'value': norm_config.enabled},
    {'area': 'normalization', 'setting': 'method', 'value': 'hip_torso'},
    {'area': 'normalization', 'setting': 'keep_reference_columns', 'value': norm_config.keep_reference_columns},
    {'area': 'normalization', 'setting': 'model_depth_scale', 'value': norm_config.model_depth_scale},
    {'area': 'normalization', 'setting': 'scale_reference', 'value': 'sequence_median_torso_length'},
    {'area': 'normalization', 'setting': 'translation_reference', 'value': 'framewise_hip_center'},
])
display(active_normalization_config)
print('PASS: normalization summary/config generated')


## Check 3: Visual Comparison


In [ ]:
recording_view_camera = dict(
    eye=dict(x=0.0, y=-2.5, z=0.0),
    center=dict(x=0.0, y=0.0, z=0.0),
    up=dict(x=0.0, y=0.0, z=1.0),
    projection=dict(type='orthographic'),
)


def apply_recording_view_camera(fig):
    fig.update_layout(scene_camera=recording_view_camera)
    return fig


fig_compare = create_pose_comparison_animation(
    df=norm_df,
    landmarks=LANDMARKS,
    connections=CONNECTIONS,
    coord_modes=("raw", "norm"),
    names=("Preprocessed", "Normalized"),
    title="Preprocessed vs Normalized Pose Coordinates",
    show_text=False,
    frame_duration=frame_duration_ms,
)

apply_recording_view_camera(fig_compare)
fig_compare.show()


## Check 4: Pipeline Integration


In [ ]:
pipe_config = build_stage_check_pipeline_config(
    exercise_id=TARGET_EXERCISE_ID,
    definitions_dir=TARGET_DEFINITIONS_DIR,
    enable_annotation=False,
    preprocessing_config=PreprocessingConfig(enabled=True),
    normalization_config=norm_config,
)

with warnings.catch_warnings(record=True):
    warnings.simplefilter("always")
    pipe_df, pipe_report = run_pipeline(
        df_annotated,
        config=pipe_config,
        landmarks=LANDMARKS,
    )

assert 'preprocessing' in pipe_report
assert 'normalization' in pipe_report
assert pipe_report['exercise_definition']['exercise_id'] == TARGET_EXERCISE_ID
assert abs(pipe_report['normalization']['scale_value'] - norm_report['scale_value']) < 1e-9
assert 'preprocessing_valid' in pipe_df.columns

pipeline_summary = pd.DataFrame([
    {'item': 'steps_executed', 'value': list(pipe_report.keys())},
    {'item': 'pipeline_exercise_id', 'value': pipe_report['exercise_definition']['exercise_id']},
    {'item': 'pipeline_output_shape', 'value': pipe_df.shape},
    {'item': 'pipeline_normalization_scale_value', 'value': round(float(pipe_report['normalization']['scale_value']), 6)},
    {'item': 'pipeline_model_depth_scale', 'value': pipe_report['normalization']['model_depth_scale']},
])
display(pipeline_summary)
print('PASS: normalization step in pipeline report')


## Check Summary

This notebook is a compact execution/QC checkpoint for ⑤ Normalization. It confirms that normalization consumes preprocessing output, preserves preprocessing provenance, and adds only the `norm` coordinate family.
